# Chapter 04 — Tkinter variables, callbacks, and application state

**Book role:** Core GUI  
**Suggested book space:** about 8 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Use Tkinter variable classes appropriately.
- Write callbacks with explicit inputs and return behavior.
- Share state among widgets without coupling data logic to controls.
- Avoid late-binding errors in callbacks created in loops.

## 1. Choosing the right variable

Use `StringVar` for text selections and entry content, `IntVar` for integer-like options, `DoubleVar` for numeric scale values, and `BooleanVar` for toggles. A variable’s `get()` returns a Python value while `set()` updates the Tcl-backed value. Conversion can fail if a field contains invalid input, so variable type does not replace validation.

A Tk variable can be observed with `trace_add`, but traces should be small and predictable. They are useful for synchronizing dependent controls or showing a live preview; they are poor places to do expensive cleaning or issue a database query after every keystroke. Debounce repeated work and keep derived business logic in a normal Python function.

### Worked example 1: Bind controls through StringVar and BooleanVar

The callback extracts plain values. In a real app, pass those values to an analysis service instead of printing them.

In [ ]:
import tkinter as tk
from tkinter import ttk

root = tk.Tk()
root.title("State and controls")
region = tk.StringVar(value="All")
include_cancelled = tk.BooleanVar(value=False)

frame = ttk.Frame(root, padding=16)
frame.pack(fill="both", expand=True)
ttk.Label(frame, text="Region").grid(row=0, column=0, sticky="w", padx=(0, 12), pady=6)
ttk.Combobox(frame, textvariable=region, values=("All", "North", "South", "East", "West"), state="readonly").grid(row=0, column=1, sticky="ew")
ttk.Checkbutton(frame, text="Include cancelled", variable=include_cancelled).grid(row=1, column=0, columnspan=2, sticky="w", pady=6)

def show_state() -> None:
    print({"region": region.get(), "include_cancelled": include_cancelled.get()})

ttk.Button(frame, text="Print state", command=show_state).grid(row=2, column=1, sticky="e", pady=(12, 0))
frame.columnconfigure(1, weight=1)
root.mainloop()

## 2. Callbacks and closure behavior

A button’s `command` expects a callable, not the result of calling that callable. `command=save_record` passes the function; `command=save_record()` runs it immediately during widget creation and passes its return value instead. Use `lambda` or `functools.partial` to supply arguments. Remember that callback functions usually run later, after the function that created the widget has returned.

Python closures capture variables, not a fresh value for each loop iteration by default. A common bug is `command=lambda: choose(i)` in a loop; each callback may observe the final value of `i`. Bind the current value with a default parameter (`lambda value=i: choose(value)`) or use `partial`. This matters when creating a dashboard with many category buttons.

### Worked example 2: Avoid callback late binding

The default argument captures the current loop value at callback creation time. `functools.partial(report, choice)` is an equally valid alternative.

In [ ]:
import tkinter as tk
from tkinter import ttk

root = tk.Tk()
frame = ttk.Frame(root, padding=12)
frame.pack()


def report(choice: str) -> None:
    print("Selected:", choice)

for choice in ("Revenue", "Profit", "Quantity"):
    ttk.Button(frame, text=choice, command=lambda value=choice: report(value)).pack(fill="x", pady=3)

root.mainloop()

## 3. Separate UI state from analytical state

A control can produce a simple filter payload such as `{"region": "North", "minimum_sales": 1000}`. A service function takes that payload and a DataFrame and returns a filtered result. The UI reads controls, calls the service, and renders the result. This separation makes filtering testable with ordinary Python unit tests and reduces the risk of having SQL conditions, `Entry.get()` calls, and Treeview updates in a single callback.

Use explicit state transitions: initial, loading, ready, empty, error. These states can drive button availability and status messages. A disabled Export button when no rows are selected is more understandable than allowing a click that silently produces an empty file.

## Guided lab: Lab: reusable filter-state object

Create a small class or dataclass for analytical filter criteria (date range, category, and region). The UI should convert validated widget values into that object. Write a separate pure function that filters a sample DataFrame from the criteria. Test the pure function without importing Tkinter.

## Review questions and practice

1. What does a Tk variable provide beyond a regular local variable?
2. Why is `command=save()` usually incorrect?
3. How can a callback created inside a loop capture the wrong value?
4. What states should a data-loading screen distinguish?

## Further reading

- Tkinter variable classes: https://docs.python.org/3/library/tkinter.html#tkinter-variable-classes
- Python closures and lambda: https://docs.python.org/3/tutorial/controlflow.html#lambda-expressions

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.